# 第 53 课｜线程、进程与回测并行边界

本 Notebook 对应「365 天量化金融」第 53 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/53_youtube-LGYrfTakPyM/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟 I/O 任务和确定性计算任务
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

先判断瓶颈是 I/O 等待还是 CPU 计算，再选择线程、进程或向量化。课程重点不是“核心越多越快”，而是任务粒度、序列化、随机种子和结果顺序。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# concurrent.futures 属于标准库；本 Notebook 实际执行线程示例。
from concurrent.futures import ThreadPoolExecutor
import time

## 1. ThreadPoolExecutor 适合 I/O 等待

每项任务仅等待 20ms。串行总时间约为任务数乘等待，线程池可在等待期间推进其他任务。性能只做现场测量，不写死加速倍数断言。

In [3]:
# 比较 8 个模拟 I/O 请求的串行与线程执行。
def io_task(symbol):
    time.sleep(0.02)
    return {"symbol": symbol, "rows": 10}

symbols = [f"ASSET_{i}" for i in range(8)]
start = time.perf_counter()
serial = [io_task(symbol) for symbol in symbols]
serial_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    threaded = list(executor.map(io_task, symbols))
thread_time = time.perf_counter() - start

print(f"串行 {serial_time:.3f}s，线程 {thread_time:.3f}s，加速比 {serial_time / thread_time:.2f}x")
assert [item["symbol"] for item in threaded] == symbols

串行 0.190s，线程 0.055s，加速比 3.42x


## 2. CPU 任务先考虑 NumPy 向量化

纯 Python 循环受 GIL 影响；NumPy 在底层数值库中批量计算。线程不能自动让 Python 字节码并行，进程也可能被传输大数组的 Pickle 开销抵消。

In [4]:
# 用相同输入比较 Python 循环与 NumPy 向量化，核对数值一致。
rng = np.random.default_rng(365)
array = rng.normal(size=300_000)

def python_sum_of_squares(values):
    total = 0.0
    for value in values:
        total += value * value
    return total

start = time.perf_counter()
python_result = python_sum_of_squares(array)
python_time = time.perf_counter() - start
start = time.perf_counter()
numpy_result = np.square(array).sum()
numpy_time = time.perf_counter() - start

print(f"Python {python_time:.4f}s，NumPy {numpy_time:.4f}s")
np.testing.assert_allclose(python_result, numpy_result, rtol=1e-12)

Python 0.0218s，NumPy 0.0004s


## 3. Amdahl 定律给并行加速设上限

若可并行比例为 `f`，使用 p 个执行单元，理论上限为：

`S_p <= 1 / ((1-f) + f/p)`。

即使无限核心，串行部分仍然存在。

In [5]:
# 比较不同并行比例和核心数的理论上限。
rows = []
for fraction in [0.50, 0.80, 0.95]:
    for workers in [2, 4, 8, 16]:
        speedup = 1 / ((1 - fraction) + fraction / workers)
        rows.append({
            "parallel_fraction": fraction,
            "workers": workers,
            "max_speedup": speedup,
            "efficiency": speedup / workers,
        })
amdahl = pd.DataFrame(rows)
display(amdahl.pivot(index="parallel_fraction", columns="workers", values="max_speedup").round(2))
assert amdahl["max_speedup"].gt(1).all()

workers,2,4,8,16
parallel_fraction,,,,
0.50,1.33,1.60,1.78,1.88
0.80,1.67,2.50,3.33,4.00
0.95,1.90,3.48,5.93,9.14


## 4. ProcessPool 的安全脚本模板

macOS/Jupyter 的进程启动语义与普通脚本不同，因此课程页不在内核中强行 fork。下面生成一个可从终端运行的独立脚本：计算函数定义在模块顶层，并用 `if __name__ == "__main__"` 保护入口。

In [6]:
# 生成独立进程池模板，不在 Notebook 内核里启动子进程。
process_script = '''from concurrent.futures import ProcessPoolExecutor

def backtest_task(seed):
    total = 0
    for value in range(200_000):
        total += (value + seed) % 97
    return seed, total

if __name__ == "__main__":
    seeds = [365, 366, 367, 368]
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(backtest_task, seeds))
    print(sorted(results))
'''
script_path = OUTPUT_DIR / "53_process_pool_example.py"
script_path.write_text(process_script, encoding="utf-8")
print(process_script)
assert 'if __name__ == "__main__":' in process_script
assert script_path.exists()

from concurrent.futures import ProcessPoolExecutor

def backtest_task(seed):
    total = 0
    for value in range(200_000):
        total += (value + seed) % 97
    return seed, total

if __name__ == "__main__":
    seeds = [365, 366, 367, 368]
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(backtest_task, seeds))
    print(sorted(results))



## 方法边界与复用结论


- 小任务并行可能更慢；先剖析热点，再决定并行层级。
- BLAS、NumPy 或模型内部可能已多线程，外层进程池会过度订阅 CPU。
- 并行任务要派生独立随机种子，并显式排序结果；完成顺序不能参与策略逻辑。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。